In [0]:
%run ./01_setup_environment

In [0]:

# ========================================
# 02_bronze_patients_ingestion
# ========================================

from pyspark.sql.functions import *

try:

    patients_df = spark.read.format("csv") \
        .option("header", True) \
        .option("inferSchema", True) \
        .load(f"{source_path}/patients")

    bronze_patients_df = patients_df.withColumn(
        "ingestion_time",
        current_timestamp()
    ).withColumn(
        "source_file",
        col("_metadata.file_path")
    )

    bronze_patients_df.write \
        .format("delta") \
        .mode("append") \
        .save(f"{bronze_path}/patients")

    log_audit(
        "patients_pipeline",
        "bronze",
        "bronze_patients",
        bronze_patients_df.count(),
        "SUCCESS"
    )

    print("Bronze Patients Load Completed")

except Exception as e:

    log_audit(
        "patients_pipeline",
        "bronze",
        "bronze_patients",
        0,
        "FAILED",
        str(e)
    )

    raise e